# Import Required Libraries
This section imports all necessary libraries for SVM training and evaluation, including numpy, pandas, scikit-learn, matplotlib, and joblib.

In [1]:
# Import libraries
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import joblib

# Load and Prepare Dataset
Load the quantum kernel matrix and the preprocessed data. Use only the first 50 samples to match the kernel matrix.

In [2]:
# Load quantum kernel matrix (remove header row)
kernel_matrix = np.loadtxt('quantum_kernel_matrix.csv', delimiter=',', skiprows=1)

# Load labels from preprocessed data (first 50 samples)
data = pd.read_csv('preprocessed data.csv').iloc[:50]
labels = data['diagnosis'].map({'M': 1, 'B': 0}).values

# Split Data into Training and Test Sets
Split the data into training and test sets using an 80-20 ratio, stratified by the labels.

In [3]:
# Train-test split (80-20)
num_samples = kernel_matrix.shape[0]
indices = np.arange(num_samples)
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=labels[:num_samples])

# Prepare kernel matrices for SVM
K_train = kernel_matrix[np.ix_(train_idx, train_idx)]
K_test = kernel_matrix[np.ix_(test_idx, train_idx)]
y_train = labels[train_idx]
y_test = labels[test_idx]

# Train SVM Model
Train the SVM classifier using the precomputed quantum kernel matrix.

In [4]:
# Train SVM with precomputed kernel
svm = SVC(kernel='precomputed')
svm.fit(K_train, y_train)

SVC(kernel='precomputed')

# Evaluate Model Performance
Evaluate the SVM model using accuracy and save the trained model for later use.

In [9]:
# Predict and calculate accuracy
y_pred = svm.predict(K_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

# Save the trained model
joblib.dump(svm, 'svm_model.joblib')

Test Accuracy: 0.9000


['svm_model.joblib']

# Visualize Results
Visualize the support vectors and the data points using the first two principal components (PC1 and PC2).

In [ ]:
# Visualize support vectors and data points
plt.figure(figsize=(8,6))
X_pc = data[['PC1', 'PC2']].values
plt.scatter(X_pc[train_idx,0], X_pc[train_idx,1], c=y_train, cmap='coolwarm', label='Train', alpha=0.6)
plt.scatter(X_pc[test_idx,0], X_pc[test_idx,1], c=y_test, cmap='coolwarm', marker='x', label='Test', alpha=0.8)

# Highlight support vectors
support_idx = train_idx[svm.support_]
plt.scatter(X_pc[support_idx,0], X_pc[support_idx,1], s=120, facecolors='none', edgecolors='k', label='Support Vectors')

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('SVM Support Vectors and Hyperplane (PC1 vs PC2)')
plt.legend()
plt.tight_layout()
plt.savefig('svm_support_vectors.png')
plt.show()

# Run SVM on Multiple Splits
Evaluate the SVM model on multiple random train-test splits and report the average accuracy.

In [ ]:
# Run SVM on multiple splits and report average accuracy
from tqdm import tqdm
n_splits = 20
accuracies = []
for seed in tqdm(range(n_splits)):
    train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=seed, stratify=labels[:num_samples])
    K_train = kernel_matrix[np.ix_(train_idx, train_idx)]
    K_test = kernel_matrix[np.ix_(test_idx, train_idx)]
    y_train = labels[train_idx]
    y_test = labels[test_idx]
    svm = SVC(kernel='precomputed')
    svm.fit(K_train, y_train)
    y_pred = svm.predict(K_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
print(f"Average accuracy over {n_splits} splits: {np.mean(accuracies):.4f}")